In [ ]:
import os
import shutil

# Create the target directory inside Colab's local runtime
os.makedirs('/content/data', exist_ok=True)

# Path to the shortcut folder in your Google Drive
drive_folder_path = '/content/drive/MyDrive/DATA_CLEAN'

# Copying the images folder
print("Copying images, please wait a moment...")
if os.path.exists(f"{drive_folder_path}/images"):
    shutil.copytree(f"{drive_folder_path}/images", "/content/data/images", dirs_exist_ok=True)
    print("Images folder successfully copied to local storage!")
else:
    print("Error: 'images' folder not found in Google Drive path.")

# Copying the labels folder
print("Copying labels...")
if os.path.exists(f"{drive_folder_path}/labels"):
    shutil.copytree(f"{drive_folder_path}/labels", "/content/data/labels", dirs_exist_ok=True)
    print("Labels folder successfully copied to local storage!")
else:
    print("rror: 'labels' folder not found in Google Drive path.")

Copying images, please wait a moment...
Images folder successfully copied to local storage!
Copying labels...
Labels folder successfully copied to local storage!


### Model Variant Selection and Justification

I have selected the **yolo26s** variant for this task. Because:
1. The dataset contains 3,327 images, which is large enough to benefit from the higher capacity and accuracy (48.6% mAP) of the 's' variant compared to the 'n' variant without underfitting.
2. I am utilizing a dedicated cloud GPU runtime in Google Colab, which can easily handle the 9.5M parameters of `yolo26s` within a reasonable training time for 30 epochs.
3. The model needs to be exported to ONNX and deployed via Docker next Friday; `yolo26s` strikes the perfect balance ("sweet spot") between state-of-the-art performance and a lightweight deployment footprint, ensuring fast inference speeds and small container sizes.

In [24]:
import os

images_dir = '/content/data/images'
files = ['train.txt', 'val.txt', 'test.txt']

for file_name in files:
    file_path = os.path.join('/content/data', file_name)

    if os.path.exists(file_path):
        with open(file_path, 'r') as f:
            lines = f.readlines()

        with open(file_path, 'w') as f:
            for line in lines:
                # Extracts only the filename and constructs the absolute path
                img_name = os.path.basename(line.strip().replace('\\', '/'))
                f.write(f"{images_dir}/{img_name}\n")
        print(f"✅ Fixed paths in {file_name}")

✅ Fixed paths in train.txt
✅ Fixed paths in val.txt
✅ Fixed paths in test.txt


In [25]:
# Check the first few lines of train.txt to verify the format
with open('/content/data/train.txt', 'r') as f:
    print("First 3 lines of train.txt:")
    print(f.read(200)) # Prints the first 200 characters

# Verify if one of these files actually exists in the directory
import os
with open('/content/data/train.txt', 'r') as f:
    sample_path = f.readline().strip()
    print(f"\nChecking existence of sample path: {sample_path}")
    print(f"Exists: {os.path.exists(sample_path)}")

First 3 lines of train.txt:
/content/data/images/39b2b787c450132e.jpg
/content/data/images/29731bc4fadc27c5.jpg
/content/data/images/11762fef71d44be3.jpg
/content/data/images/9c2fbb150200b2e8.jpg
/content/data/images/393c19ad51a

Checking existence of sample path: /content/data/images/39b2b787c450132e.jpg
Exists: True


In [26]:
import os
from ultralytics import YOLO

# Ensure any stale cache is removed to force a fresh scan
cache_files = ['/content/data/train.cache', '/content/data/val.cache', '/content/data/test.cache']
for cache in cache_files:
    if os.path.exists(cache):
        os.remove(cache)

# Start the training process
model = YOLO("yolo26s.pt")

model.train(
    data="/content/data.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    project="runs",
    name="cats_v1",
    seed=42
)

Ultralytics 8.4.52 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=cats_v1-9, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspe

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7ba7bde0c230>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 